# crop-mesh-detector on Colab (Stage 1: local-only training)

Runs **stage 1** of the two-stage pipeline on a free NVIDIA GPU instead of your laptop's
CPU: local-only supervised training per node, no knowledge exchange. PlantVillage and
PlantDoc (the merged project dataset -- see `src/data/merged.py`) are fetched straight
from their sources on Colab's fast network -- no manual data transfer.

Stage 1 writes its checkpoints, manifest, and results to `outputs/stage1_local/`.
Once you've downloaded and inspected those results, **stage 2** (`src/train_mesh.py`,
the mesh prototype + probe-logit exchange, warm-started from stage 1's checkpoints) is a
separate run -- this notebook does not run it.

**Trains each of the 3 architectures as a separate step.** Colab's free tier caps GPU usage,
so training all 3 in one long run risks losing everything to a disconnect partway through.
Instead, each architecture's cell trains, exports a Raspberry-Pi-ready bundle for *that*
architecture, and downloads its results immediately -- so even if the session disconnects
before you get to the next architecture, whatever finished is already safely on your Mac.
Results accumulate into one combined `outputs/stage1_local/results_summary.json` as you go
(as long as the session stays connected between cells), so the final comparison + bulk
export still covers all 3.

**Before running anything**: `Runtime` menu -> `Change runtime type` -> Hardware accelerator -> `T4 GPU` -> Save.

In [ ]:
# Confirm the GPU runtime is actually attached
!nvidia-smi

## 1. Clone the project (private repo)

This repo is private, so cloning it here needs a GitHub personal access token --
generate one at github.com -> Settings -> Developer settings -> Personal access tokens
(fine-grained, read-only access to this one repo is enough). `getpass` keeps it out of
the notebook's saved output/history.

Clones the `kb-tuning-02` branch specifically -- `src/train_local.py`/`src/train_mesh.py`
(the two-stage pipeline this notebook runs) only exist there, not on `main` yet.

In [ ]:
from getpass import getpass

token = getpass('GitHub personal access token: ')
!git clone --branch kb-tuning-02 https://{token}@github.com/karboon1008/crop-mesh-detector.git /content/crop-mesh-detector
del token
%cd /content/crop-mesh-detector
!ls

## 2. Install dependencies

Colab already ships recent `torch`/`tensorflow`/`numpy`; this adds what's missing
(`timm`, `codecarbon`, `onnx`, `onnxruntime`) and pins the rest per `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "protobuf>=5.29.1,<6"

## 3. Fetch PlantVillage + PlantDoc

Same scripts as local. Stage 1 trains on the merged PlantVillage+PlantDoc project dataset
(`src/data/merged.py`), so both need to be present. PlantVillage comes via the
`tensorflow_datasets` library catalog entry (no Kaggle account needed) and is written out as
`.jpg` files under `data/PlantVillage/`; PlantDoc is fetched from its GitHub release into
`data/PlantDoc/`. Takes a few minutes on Colab's network. Shared by all 3 architectures
below -- only needs to run once per session.

In [ ]:
!python scripts/download_plantvillage.py
!python scripts/download_plantdoc.py

## 4. Model 1/3: mobilenet_v3_small (stage 1: local-only)

`--arch` overrides `config.yaml`'s architecture list for this one run. `src/train_local.py`
picks `cuda` automatically when available -- on Colab's GPU runtime that resolves to
the T4, no code change needed. Results merge into `outputs/stage1_local/results_summary.json`
(`--fresh` here means: start a brand-new stage-1 run, discarding any old results/manifest
from a previous session).

In [ ]:
!python -m src.train_local --config config.yaml --arch mobilenet_v3_small --fresh

In [ ]:
# Export this architecture's best node to a Pi-ready bundle, then download its
# results immediately -- protects this model's results even if the next one fails.
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage1_local/checkpoints --results outputs/stage1_local/results_summary.json --arch mobilenet_v3_small --node node_0 --output-dir outputs/pi_export/mobilenet_v3_small

import shutil
from google.colab import files

shutil.make_archive('/content/mobilenet_v3_small_bundle', 'zip', 'outputs/pi_export/mobilenet_v3_small')
files.download('/content/mobilenet_v3_small_bundle.zip')

## 5. Model 2/3: efficientnet_lite0

No `--fresh` this time -- merges into the same `outputs/stage1_local/results_summary.json` from step 4.

In [ ]:
!python -m src.train_local --config config.yaml --arch efficientnet_lite0

In [ ]:
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage1_local/checkpoints --results outputs/stage1_local/results_summary.json --arch efficientnet_lite0 --node node_0 --output-dir outputs/pi_export/efficientnet_lite0

shutil.make_archive('/content/efficientnet_lite0_bundle', 'zip', 'outputs/pi_export/efficientnet_lite0')
files.download('/content/efficientnet_lite0_bundle.zip')

## 6. Model 3/3: mobilevit_xxs

In [ ]:
!python -m src.train_local --config config.yaml --arch mobilevit_xxs

In [ ]:
!python scripts/export_for_pi.py --checkpoints-dir outputs/stage1_local/checkpoints --results outputs/stage1_local/results_summary.json --arch mobilevit_xxs --node node_0 --output-dir outputs/pi_export/mobilevit_xxs

shutil.make_archive('/content/mobilevit_xxs_bundle', 'zip', 'outputs/pi_export/mobilevit_xxs')
files.download('/content/mobilevit_xxs_bundle.zip')

## 7. Compare all 3 and download the combined stage-1 results

Only meaningful if the session stayed connected through steps 4-6 (so
`outputs/stage1_local/results_summary.json` actually has all 3 architectures merged in).
`--all` re-exports every architecture's best node into its own subfolder -- redundant with
steps 4/5/6 above, but convenient as one combined bundle covering all 3 to compare on the Pi.

Stage 1 alone has no sustainability report (that's produced by stage 2, which combines
compute + communication cost) -- `run_state.json` here has stage 1's accumulated compute
energy/duration instead. Once you've looked over these results and are happy with them,
download `outputs/stage1_local/` (below) and run stage 2 separately:
`python -m src.train_mesh --config config.yaml`.

In [ ]:
!python scripts/export_for_pi.py --all --checkpoints-dir outputs/stage1_local/checkpoints --results outputs/stage1_local/results_summary.json
!cat outputs/stage1_local/results_summary.json
!cat outputs/stage1_local/run_state.json

In [ ]:
shutil.make_archive('/content/stage1_local', 'zip', 'outputs/stage1_local')
files.download('/content/stage1_local.zip')